# for evaluating dynamic deephit performance (e.g. c-index, brier score, etc)

In [58]:
import sys
import os
from pathlib import Path
import importlib
import json

import numpy as np
import matplotlib.pyplot as plt
import torch

# Optional but very useful in notebooks
%load_ext autoreload
%autoreload 2

repo_root = Path("/home/jake/Documents/AI_Clinician/ai_clinician_pediatric_rl")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.supervised_learning.dynamic_deephit_evaluation as dde
import src.supervised_learning.predict as predict

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# load dataset

In [66]:
data_dir = "/home/jake/Documents/AI_Clinician/choc_data/combined_data/hourly_all_patients/split_0/training_set"
dynamic_deephit_dataset = dde.load_dynamic_deephit_dataset(model_dir=data_dir, split='test')

print(f"keys in dynamic_deephit_dataset: {list(dynamic_deephit_dataset.keys())}")

# expand events so we have an event label at every time step using index_ptr to determine the start and end of each trajectory
dynamic_deephit_dataset = dde.expand_event_labels(dynamic_deephit_dataset)

keys in dynamic_deephit_dataset: ['stay_ids', 'trajectories', 'index_ptr', 'feature_names', 'physiological_feature_ind', 'physiological_feature_type', 'feature_dim', 'response_zero_ind', 'carry_forward_ind', 'event', 'time_to_event_raw', 'time_to_event']


# load model predictions
## model inference was run using predict.py
### should probably consolidate the code...

In [69]:
model_dir = "/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/final_model/unknown_cx3-20-1_20260821-202508_49e52257"

dynamic_deephit_predictions = dde.load_dynamic_deephit_predictions(
    model_dir, 
    split='test',
    keys_to_exclude=['compressed_latents']
)
print(f"keys in dynamic_deephit_predictions: {list(dynamic_deephit_predictions.keys())}")

logits = dynamic_deephit_predictions['predictions']

# convert the prediction logits to joint probabilities using softmax
probabilities = dde.calculate_probabilities(logits)

keys in dynamic_deephit_predictions: ['predictions', 'stay_ids', 'index_ptr', 'model_outputs', 'health_score', 'entropy', 'carry_forward_ind', 'response_zero_ind']


In [70]:
# confirm that number of rows in predictions matches number of rows in trajectories
assert dynamic_deephit_predictions['predictions'].shape[0] == dynamic_deephit_dataset['trajectories'].shape[0], "Number of rows in predictions does not match number of rows in trajectories"
print(f"Equal number of rows in predictions and trajectories: {dynamic_deephit_predictions['predictions'].shape[0]}")

# make sure the order of stay_ids in predictions matches the order of stay_ids in trajectories
assert np.array_equal(dynamic_deephit_predictions['stay_ids'], dynamic_deephit_dataset['stay_ids']), "Order of stay_ids in predictions does not match order of stay_ids in trajectories"
print("Order of stay_ids in predictions matches order of stay_ids in trajectories")

Equal number of rows in predictions and trajectories: 499032
Order of stay_ids in predictions matches order of stay_ids in trajectories


In [76]:
time_to_event = torch.as_tensor(
    dynamic_deephit_dataset["time_to_event"],
    dtype=torch.long,
)
event_labels = torch.as_tensor(
    dynamic_deephit_dataset["event_labels"],
    dtype=torch.long,
)
logits = torch.as_tensor(logits, dtype=torch.float32)

survival_metrics = dde.compute_survival_metrics(
            logits,
            time_to_event,
            event_labels,
            return_full_curves=True,
            no_censoring=False
        )

In [78]:
print(survival_metrics)

{'weighted_C_index': tensor([0.7047, 0.9392]), 'weighted_multi_C': 0.7162839770317078, 'ibs_weighted': tensor([0.1390, 0.0158]), 'auroc': 0.9577317104511291, 'average_precision': 0.7883976803825647, 'prevalence': 0.04977436316709149, 'roc_curve': {'fpr': array([0.        , 0.        , 0.        , ..., 0.99699279, 0.996997  ,
       1.        ]), 'tpr': array([0.00000000e+00, 4.02592697e-05, 6.52200169e-03, ...,
       1.00000000e+00, 1.00000000e+00, 1.00000000e+00]), 'thresholds': array([       inf, 0.9904585 , 0.9879913 , ..., 0.00563293, 0.0056329 ,
       0.00416157], dtype=float32)}, 'pr_curve': {'precision': array([0.04977436, 0.04977446, 0.04977456, ..., 1.        , 1.        ,
       1.        ]), 'recall': array([1.00000000e+00, 1.00000000e+00, 1.00000000e+00, ...,
       8.05185394e-05, 4.02592697e-05, 0.00000000e+00]), 'thresholds': array([0.00416157, 0.0042153 , 0.00422552, ..., 0.98978966, 0.98979247,
       0.9904585 ], dtype=float32)}}
